# EA-TTM — Project Bootstrap and Environment Audit

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 00 — Reproducible project bootstrap  
**Scope:** Establish the Google Colab runtime, deterministic execution controls, persistent project structure, and a machine-readable environment manifest.

This notebook does **not** select or train a forecasting model. Model-selection decisions are intentionally deferred until the TTM model contract is audited in the next phase.

## Experimental lineage

The implementation will distinguish three forecasting systems:

$$
\mathcal{M}_{ZS}
\;\longrightarrow\;
\mathcal{M}_{HP}
\;\longrightarrow\;
\mathcal{M}_{EA}
$$

where:

- $\mathcal{M}_{ZS}$: **Zero-shot TTM** — the complete pre-trained TTM is used without target-domain training.
- $\mathcal{M}_{HP}$: **Head-Probed TTM** — the pre-trained TTM backbone is frozen while the Slim TTM Decoder and Forecasting Head are trained on the retail target domain. The resulting TTM is then frozen.
- $\mathcal{M}_{EA}$: **EA-TTM** — the frozen head-probed TTM supplies the base forecast and Slim Decoder representation to the trainable shock-aware residual adapter.

The primary EA-TTM equation is:

$$
\hat{y}_{EA,h}
=
\hat{y}_{HP,h}
+
g_h\,\Delta_h
$$

During adapter training, the head-probed TTM is treated as the fixed base forecaster.


## Reproducibility contract

Let $s$ denote the experiment seed. Deterministic state is initialized consistently for Python, NumPy, PyTorch, and CUDA:

$$
\begin{aligned}
s_{\mathrm{Python}}
&=
s_{\mathrm{NumPy}}
=
s_{\mathrm{PyTorch}}
=
s_{\mathrm{CUDA}}
=
s
\end{aligned}
$$

The bootstrap phase records:

- Python and operating-system metadata;
- PyTorch and CUDA versions;
- GPU type and CUDA capability;
- installed package versions;
- deterministic seed;
- persistent project root;
- a timestamped environment manifest.

The environment manifest is treated as an immutable provenance artifact for later handover.


In [1]:
%pip install -q "granite-tsfm==0.3.9" "pyyaml>=6.0" "pyarrow>=15.0" "psutil>=5.9"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 87.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


In [2]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import psutil
import torch
import yaml


## Project constants

The project root is the only path established in this phase. All subsequent notebooks will resolve paths from this root rather than embedding repeated absolute paths.

The random seed $s=2026$ is a project-level reproducibility constant and must not be changed between comparable runs unless a deliberate multi-seed experiment is being conducted.

In [3]:
PROJECT_NAME = "EA_TTM_RETAIL"
BASE_SEED = 2026

# Persistent storage root for the complete project state.
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME

# Core directory contract. New folders may be added later, but these names remain stable.
PROJECT_DIRECTORIES = (
    "configs",
    "data/raw",
    "data/interim",
    "data/processed",
    "notebooks",
    "src/data",
    "src/models",
    "src/training",
    "src/evaluation",
    "src/utils",
    "runs",
    "results/tables",
    "results/figures",
    "metadata",
    "docs",
    "report",
)


In [4]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for relative_directory in PROJECT_DIRECTORIES:
    (PROJECT_ROOT / relative_directory).mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [5]:
def set_global_seed(seed: int) -> None:
    """Initialize deterministic pseudo-random states used by the project."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic algorithms are preferred for reproducible experiments.
    # warn_only=True prevents unsupported deterministic kernels from terminating this audit.
    torch.use_deterministic_algorithms(True, warn_only=True)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_global_seed(BASE_SEED)


## Runtime resource audit

The compute audit verifies that the runtime can execute the project on a CUDA device and records the device characteristics required for reproducible handover.

No model-size or batch-size decision is made here.

In [6]:
def command_output(command: list[str]) -> str | None:
    """Execute a command and return stripped stdout; return None if unavailable."""
    try:
        completed = subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
        )
        return completed.stdout.strip()
    except Exception:
        return None


cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
gpu_capability = (
    list(torch.cuda.get_device_capability(0)) if cuda_available else None
)
gpu_memory_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 3)
    if cuda_available
    else None
)

runtime_resource_summary = {
    "cuda_available": cuda_available,
    "gpu_name": gpu_name,
    "gpu_compute_capability": gpu_capability,
    "gpu_memory_gb": gpu_memory_gb,
    "cpu_logical_count": psutil.cpu_count(logical=True),
    "system_memory_gb": round(psutil.virtual_memory().total / (1024**3), 3),
    "nvidia_smi": command_output(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"]
    ),
}

runtime_resource_summary


{'cuda_available': True,
 'gpu_name': 'Tesla T4',
 'gpu_compute_capability': [7, 5],
 'gpu_memory_gb': 14.563,
 'cpu_logical_count': 2,
 'system_memory_gb': 12.671,
 'nvidia_smi': 'Tesla T4, 580.82.07, 15360 MiB'}

In [7]:
assert cuda_available, (
    "CUDA GPU is not available. EA-TTM training must not begin from this runtime."
)

# Lightweight device-level computation check.
device = torch.device("cuda")
probe_tensor = torch.arange(16, dtype=torch.float32, device=device).reshape(4, 4)
probe_checksum = float((probe_tensor @ probe_tensor.T).sum().item())

assert np.isfinite(probe_checksum)


## Package and environment provenance

Package versions are captured from the runtime **after** the project dependencies have been installed. This manifest is later used to reproduce the environment on another Colab account.

A SHA-256 digest is computed over the normalized manifest so that later notebooks can identify whether their runtime differs from the bootstrap environment.

In [8]:
def package_version(package_name: str) -> str | None:
    """Read an installed distribution version without importing optional packages."""
    from importlib.metadata import PackageNotFoundError, version

    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


tracked_packages = [
    "granite-tsfm",
    "torch",
    "transformers",
    "huggingface-hub",
    "numpy",
    "pandas",
    "scikit-learn",
    "pyarrow",
    "pyyaml",
    "psutil",
]

package_versions = {
    package_name: package_version(package_name)
    for package_name in tracked_packages
}

package_versions


{'granite-tsfm': '0.3.9',
 'torch': '2.11.0+cu128',
 'transformers': '5.16.1',
 'huggingface-hub': '1.29.0',
 'numpy': '2.1.3',
 'pandas': '3.0.6',
 'scikit-learn': '1.6.1',
 'pyarrow': '23.0.1',
 'pyyaml': '6.0.3',
 'psutil': '5.9.5'}

In [9]:
bootstrap_timestamp = datetime.now(timezone.utc).isoformat()

environment_manifest: dict[str, Any] = {
    "project_name": PROJECT_NAME,
    "phase": "00_PROJECT_BOOTSTRAP",
    "created_utc": bootstrap_timestamp,
    "project_root": str(PROJECT_ROOT),
    "seed": BASE_SEED,
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "cuda_runtime_version": torch.version.cuda,
    "resource_summary": runtime_resource_summary,
    "package_versions": package_versions,
}

manifest_payload = json.dumps(
    environment_manifest,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

environment_manifest["manifest_sha256"] = hashlib.sha256(manifest_payload).hexdigest()

manifest_path = PROJECT_ROOT / "metadata" / "00_ENVIRONMENT_AUDIT.json"
manifest_path.write_text(
    json.dumps(environment_manifest, indent=2),
    encoding="utf-8",
)


1103

In [10]:
# Capture the complete runtime package state for later reconstruction.
requirements_path = PROJECT_ROOT / "metadata" / "00_REQUIREMENTS_RUNTIME.txt"

freeze_output = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=True,
    capture_output=True,
    text=True,
).stdout

requirements_path.write_text(freeze_output, encoding="utf-8")


14117

## Project-state initialization

`PROJECT_STATUS.md` is the human-readable handover record. It complements the machine-readable JSON manifest and is updated after each completed phase.

In [11]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_text = f"""# EA-TTM Project Status

## Current stage
Phase 00 — Project bootstrap completed.

## Persistent project root
`{PROJECT_ROOT}`

## Environment
- Base seed: `{BASE_SEED}`
- CUDA available: `{cuda_available}`
- GPU: `{gpu_name}`
- PyTorch: `{torch.__version__}`
- granite-tsfm: `{package_versions.get("granite-tsfm")}`

## Completed artifacts
- `metadata/00_ENVIRONMENT_AUDIT.json`
- `metadata/00_REQUIREMENTS_RUNTIME.txt`

## Model state
No TTM model has been selected or trained in Phase 00.

## Next phase
Audit and lock the exact TTM R2.1 model contract, including context length, forecast horizon,
resolved model revision, module structure, trainable-component policy, and Slim Decoder interface.
"""

status_path.write_text(status_text, encoding="utf-8")


661

## Bootstrap validation

The phase is complete only if the following reproducibility conditions hold:

$$
\begin{aligned}
&\text{CUDA available}
\\
&\land\ \text{persistent project root exists}
\\
&\land\ \text{environment manifest exists}
\\
&\land\ \text{runtime lock exists}
\end{aligned}
$$


In [12]:
required_artifacts = [
    manifest_path,
    requirements_path,
    status_path,
]

validation = {
    "project_root_exists": PROJECT_ROOT.exists(),
    "required_directories_exist": all(
        (PROJECT_ROOT / relative_directory).exists()
        for relative_directory in PROJECT_DIRECTORIES
    ),
    "required_artifacts_exist": all(path.exists() for path in required_artifacts),
    "cuda_available": cuda_available,
    "gpu_probe_checksum": probe_checksum,
}

assert all(
    [
        validation["project_root_exists"],
        validation["required_directories_exist"],
        validation["required_artifacts_exist"],
        validation["cuda_available"],
    ]
), f"Bootstrap validation failed: {validation}"

validation


{'project_root_exists': True,
 'required_directories_exist': True,
 'required_artifacts_exist': True,
 'cuda_available': True,
 'gpu_probe_checksum': 3680.0}

## Return block

The final cell emits the exact information required for the next implementation decision. No training result is produced in this phase.

In [13]:
return_block = {
    "status": "PHASE_00_COMPLETE",
    "project_root": str(PROJECT_ROOT),
    "seed": BASE_SEED,
    "cuda_available": cuda_available,
    "gpu_name": gpu_name,
    "gpu_memory_gb": gpu_memory_gb,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_runtime_version": torch.version.cuda,
    "granite_tsfm_version": package_versions.get("granite-tsfm"),
    "transformers_version": package_versions.get("transformers"),
    "manifest_sha256": environment_manifest["manifest_sha256"],
    "artifacts": {
        "environment_manifest": str(manifest_path),
        "requirements_runtime": str(requirements_path),
        "project_status": str(status_path),
    },
}

print("PHASE 00 RETURN BLOCK")
print(json.dumps(return_block, indent=2))


PHASE 00 RETURN BLOCK
{
  "status": "PHASE_00_COMPLETE",
  "project_root": "/content/drive/MyDrive/EA_TTM_RETAIL",
  "seed": 2026,
  "cuda_available": true,
  "gpu_name": "Tesla T4",
  "gpu_memory_gb": 14.563,
  "python_version": "3.13.15",
  "torch_version": "2.11.0+cu128",
  "cuda_runtime_version": "12.8",
  "granite_tsfm_version": "0.3.9",
  "transformers_version": "5.16.1",
  "manifest_sha256": "b3cb885b1be7c026aaf41ee41eba979e5e06b17d1fdefa3231616bb8f4cf59c4",
  "artifacts": {
    "environment_manifest": "/content/drive/MyDrive/EA_TTM_RETAIL/metadata/00_ENVIRONMENT_AUDIT.json",
    "requirements_runtime": "/content/drive/MyDrive/EA_TTM_RETAIL/metadata/00_REQUIREMENTS_RUNTIME.txt",
    "project_status": "/content/drive/MyDrive/EA_TTM_RETAIL/PROJECT_STATUS.md"
  }
}
